In [ ]:
"""
Parte4_carteiras.py -- 4.1: o erro de avaliacao preve retornos futuros?

Para cada alvo e cada mes t:
  1. ordena as acoes por epsilon_t (erro_pct_vm) e divide em N_QUANTIS (Q1 = mais caras, QN = mais baratas);
  2. mede o retorno de cada carteira em t+1 (ponderado por VM_t e igualmente ponderado);
  3. long-short = QN - Q1.
Depois: media mensal, t de Newey-West, Sharpe anualizado, alfas (CAPM e NEFIN) e maximo drawdown.
Nao aplica o filtro P10 da Parte 1 sobre os retornos: toda acao com epsilon em t entra com o retorno que tiver em t+1.
Saidas em resultados_parte4/: carteiras_series.parquet, carteiras_resumo.csv, carteiras_quintis.csv,
carteiras_drawdown.csv, carteiras_cobertura.csv
"""
import numpy as np
import pandas as pd
import statsmodels.api as sm

import comum_parte4 as C


def atribui_quantis(df, n=C.N_QUANTIS, breakpoints=C.BREAKPOINTS):
    """Quantil 1..n de epsilon dentro de cada (alvo, data). Com breakpoints='grandes', os cortes sao calculados
    so nas acoes acima da mediana de VM do mes (analogo aos breakpoints NYSE) e aplicados a todas."""
    if breakpoints not in ("todas", "grandes"):
        raise ValueError(breakpoints)
    df = df.copy()
    df["quantil"] = 0
    for _, g in df.groupby(["alvo", "data"]):
        base = g if breakpoints == "todas" else g[g["valor_mercado"] >= g["valor_mercado"].median()]
        cortes = np.quantile(base["erro_pct_vm"], np.linspace(0, 1, n + 1)[1:-1])
        df.loc[g.index, "quantil"] = np.searchsorted(cortes, g["erro_pct_vm"].to_numpy(), side="right") + 1
    return df


def anexa_retorno_t1(df, retornos):
    """Retorno do mes SEGUINTE a formacao: chave (ticker, data + 1 mes). Nunca o retorno do proprio mes t."""
    r = retornos.copy()
    r["data"] = pd.to_datetime(r["data"]) + pd.offsets.MonthEnd(0)
    r = r.rename(columns={"data": "data_ret"})
    df = df.copy()
    df["data_ret"] = df["data"] + pd.offsets.MonthEnd(1)
    return df.merge(r[["ticker", "data_ret", "ret"]], on=["ticker", "data_ret"], how="left")


def retornos_carteiras(df, n=C.N_QUANTIS, pctl_ew=C.FILTRO_EW_PCTL_VM):
    """Serie mensal (data de formacao t) do retorno de cada quantil, VW e EW, e do long-short."""
    d = df.dropna(subset=["ret"]).copy()
    lim_ew = d.groupby(["alvo", "data"])["valor_mercado"].transform(lambda s: s.quantile(pctl_ew))
    d["ew_ok"] = d["valor_mercado"] >= lim_ew
    vw = (d.assign(rw=d["ret"] * d["valor_mercado"])
          .groupby(["alvo", "data", "quantil"])[["rw", "valor_mercado"]].sum())
    vw = (vw["rw"] / vw["valor_mercado"]).rename("ret")
    ew = d[d["ew_ok"]].groupby(["alvo", "data", "quantil"])["ret"].mean()
    n_acoes = d.groupby(["alvo", "data", "quantil"]).size().rename("n")
    out = []
    for pond, s in (("VW", vw), ("EW", ew)):
        w = s.unstack("quantil").reindex(columns=range(1, n + 1))
        w["LS"] = w[n] - w[1]
        w.columns = w.columns.astype(str)
        w = w.reset_index().melt(id_vars=["alvo", "data"], var_name="quantil", value_name="ret")
        w["ponderacao"] = pond
        out.append(w)
    s = pd.concat(out, ignore_index=True)
    s["quantil"] = s["quantil"].astype(str)
    return s, n_acoes.reset_index()


def newey_west(y, X=None, lags=C.LAGS_NEWEY_WEST):
    y = pd.Series(y).astype(float)
    X = np.ones((len(y), 1)) if X is None else sm.add_constant(np.asarray(X, float), has_constant="add")
    ok = np.isfinite(y.to_numpy()) & np.isfinite(X).all(axis=1)
    m = sm.OLS(y.to_numpy()[ok], X[ok]).fit(cov_type="HAC", cov_kwds={"maxlags": lags})
    return m.params[0], m.tvalues[0], int(ok.sum())


def max_drawdown(ret, datas):
    """Maior queda do patrimonio acumulado, do pico ate o vale: min_t (W_t / max_{s<=t} W_s) - 1, com W_t = prod(1 + r).
    O patrimonio comeca em 1 antes do primeiro mes, entao uma perda logo no inicio tambem conta.
    Devolve (drawdown, mes do pico, mes do vale, mes em que o pico foi recuperado ou NaT se ainda nao recuperou)."""
    r = pd.Series(np.asarray(ret, float), index=pd.DatetimeIndex(datas)).dropna()
    if r.empty:
        return np.nan, pd.NaT, pd.NaT, pd.NaT
    w = (1 + r).cumprod()
    pico = np.maximum.accumulate(np.r_[1.0, w.to_numpy()])[1:]
    dd = w.to_numpy() / pico - 1
    i_vale = int(np.argmin(dd))
    if dd[i_vale] >= 0:
        return 0.0, pd.NaT, pd.NaT, pd.NaT
    antes = np.r_[1.0, w.to_numpy()][: i_vale + 2]           # patrimonio ate o vale, com o 1 inicial na posicao 0
    i_pico = int(np.argmax(antes)) - 1                      # -1 = pico no patrimonio inicial
    data_pico = r.index[i_pico] if i_pico >= 0 else r.index[0] - pd.offsets.MonthEnd(1)
    depois = np.flatnonzero(w.to_numpy()[i_vale:] >= pico[i_vale])
    data_rec = r.index[i_vale + depois[0]] if len(depois) else pd.NaT
    return float(dd[i_vale]), data_pico, r.index[i_vale], data_rec


def resumo(series, fatores):
    """Para cada (alvo, ponderacao): media e t do LS, Sharpe anualizado, alfas, maximo drawdown.
    Excesso sobre rf so nas pernas long; o drawdown usa o retorno da carteira sem descontar o rf."""
    s = series.copy()
    s["data_ret"] = s["data"] + pd.offsets.MonthEnd(1)       # fatores do mes em que o retorno acontece (t+1)
    if fatores is None:
        s["rf"] = 0.0
    else:
        f = fatores.copy()
        f["data"] = pd.to_datetime(f["data"]) + pd.offsets.MonthEnd(0)
        s = s.merge(f.rename(columns={"data": "data_ret"}), on="data_ret", how="left")
    linhas = []
    for (alvo, pond, q), g in s.groupby(["alvo", "ponderacao", "quantil"]):
        g = g.dropna(subset=["ret"]).sort_values("data")
        y = g["ret"] if q == "LS" else g["ret"] - g["rf"]     # LS e autofinanciada; pernas em excesso do rf
        media, t, nobs = newey_west(y)
        lin = dict(alvo=alvo, ponderacao=pond, carteira=q, meses=nobs, media_mensal=media, t_media=t,
                   sharpe_anual=y.mean() / y.std(ddof=1) * np.sqrt(12) if y.std(ddof=1) > 0 else np.nan)
        dd, pico, vale, rec = max_drawdown(g["ret"], g["data_ret"])      # datas = mes em que o retorno acontece
        lin.update(max_drawdown=dd, dd_pico=pico, dd_vale=vale, dd_recuperacao=rec,
                   dd_meses_queda=(vale.to_period("M") - pico.to_period("M")).n if pd.notna(vale) else np.nan)
        for nome, cols in C.FATORES_MODELO.items():
            if all(c in g for c in cols) and g[cols].notna().all(axis=1).sum() >= 24:   # meses sem fator ficam de fora
                a, ta, _ = newey_west(y, g[cols])
                lin[f"alfa_{nome}"], lin[f"t_alfa_{nome}"] = a, ta
        linhas.append(lin)
    return pd.DataFrame(linhas)


def cobertura(df):
    """Quantas obs com epsilon ficam sem retorno em t+1 (deslistagem sem retorno, dado ausente)."""
    g = df.groupby(["alvo", df["data"].dt.year])
    return pd.DataFrame({"obs_com_epsilon": g.size(), "sem_retorno_t1": g["ret"].apply(lambda s: s.isna().sum())}).reset_index()


def main():
    C.PASTA_SAIDA.mkdir(parents=True, exist_ok=True)
    prev = C.ler_previsoes()
    prev["erro_pct_vm"] = prev[C.COL_EPSILON]
    prev = prev[np.isfinite(prev["erro_pct_vm"]) & (prev["valor_mercado"] > 0)]
    if C.EXCLUI_VM_PREV_NAO_POSITIVO and "vm_prev_nao_positivo" in prev:
        prev = prev[~prev["vm_prev_nao_positivo"].astype(bool)]
    try:                                              # emissor do painel da Parte 1 (CNPJ), se existir
        em = C.ler_painel(["ticker", "data", "emissor"])
        prev = prev.merge(em, on=["ticker", "data"], how="left")
    except Exception as e:                            # sem emissor no painel: heuristica do ticker
        print("emissor do painel indisponivel, usando 4 letras do ticker:", type(e).__name__)
    prev = C.colapsa_emissor(prev)
    if C.PRESENCA_MINIMA is not None:                 # filtro de liquidez com informacao ate t
        prev = C.filtra_liquidez(prev, C.carrega_liquidez())
        fora = prev.loc[~prev["liquida"]]
        prev = prev.loc[prev["liquida"]]
        print(f"filtro de liquidez (presenca >= {C.PRESENCA_MINIMA:.0%} dos pregoes em {C.JANELA_LIQUIDEZ_MESES}m): "
              f"{len(fora)} obs removidas, {len(prev)} mantidas")
        fora.groupby(["alvo", fora["data"].dt.year]).size().rename("removidas").to_csv(
            C.PASTA_SAIDA / "carteiras_filtro_liquidez.csv")
    df = anexa_retorno_t1(atribui_quantis(prev), C.carrega_retornos())
    cob = cobertura(df)
    print("obs sem retorno em t+1 (descartadas):", int(cob["sem_retorno_t1"].sum()), "de", int(cob["obs_com_epsilon"].sum()))
    series, n_acoes = retornos_carteiras(df)
    tab = resumo(series, C.carrega_fatores())
    n_med = n_acoes.groupby(["alvo", "quantil"])["n"].mean().unstack().round(1)

    series.to_parquet(C.PASTA_SAIDA / "carteiras_series.parquet")
    tab.to_csv(C.PASTA_SAIDA / "carteiras_resumo.csv", index=False)
    cob.to_csv(C.PASTA_SAIDA / "carteiras_cobertura.csv", index=False)
    n_med.to_csv(C.PASTA_SAIDA / "carteiras_n_acoes.csv")

    pd.set_option("display.width", 220, "display.max_columns", 30)
    cols_dd = ["max_drawdown", "dd_pico", "dd_vale", "dd_recuperacao", "dd_meses_queda"]
    ls = tab[tab["carteira"] == "LS"].copy()
    for c in [c for c in ("media_mensal", "alfa_CAPM", "alfa_NEFIN5", "max_drawdown") if c in ls]:
        ls[c] = (ls[c] * 100).round(2)
    print("\nLong-short QN - Q1 (media, alfas e max. drawdown em %):")
    print(ls.drop(columns=["carteira"] + cols_dd[1:]).round(2).to_string(index=False))
    dd = tab.pivot_table(index=["alvo", "ponderacao"], columns="carteira", values="max_drawdown") * 100
    dd.round(1).to_csv(C.PASTA_SAIDA / "carteiras_drawdown.csv")
    print("\nMaximo drawdown por carteira (% do pico ao vale, retorno sem descontar o rf):")
    print(dd.round(1).to_string())
    print("\nPeriodo do maximo drawdown da long-short:")
    per = tab.loc[tab["carteira"] == "LS", ["alvo", "ponderacao"] + cols_dd].copy()
    per["max_drawdown"] = (per["max_drawdown"] * 100).round(1)
    for c in ("dd_pico", "dd_vale", "dd_recuperacao"):
        per[c] = pd.to_datetime(per[c]).dt.strftime("%Y-%m").fillna("nao recuperou")
    print(per.to_string(index=False))
    quint = tab[tab["carteira"] != "LS"].pivot_table(index=["alvo", "ponderacao"], columns="carteira", values="media_mensal") * 100
    quint.round(2).to_csv(C.PASTA_SAIDA / "carteiras_quintis.csv")
    print("\nExcesso de retorno medio por quantil (% ao mes; Q1 = caras, Q5 = baratas):")
    print(quint.round(2).to_string())
    print("\nAcoes por quantil (media mensal):")
    print(n_med.to_string())


if __name__ == "__main__":
    main()